# Create Dataframe Using Wendt et al. (2022) Data
This notebook contains the complete workthrough for creating the dataframe wendt_df, which is used primarily for Principal Component Analysis and machine learning in subquestion 2, and is further analyzed using remote sensing in subquestion 3. This is a separate notebook to separate the data cleaning and data engineering methods used from the machine learning aspects.

# Import Necessary Packages

In [1]:
import pandas as pd
import numpy as np

In [2]:
wendt = pd.read_csv('wendt_data.csv', skiprows=3)
wendt

,Grid FID,Grid ID,Longitude\n(degrees),Latitude\n(degrees),CO2 Source ID (tied to NATCARB dataset)*,Distance (km),CO2 Source ID (tied to NATCARB dataset),Distance (km).1,CO2 Source ID (tied to NATCARB dataset).1,Distance (km).2,...,Sum of Reservoir Quality W/O Depth (feet),New Injectivity,S1,S2,S3,S4,S1.1,S2.1,S3.1,S4.1
0,0,36663,-96.964053,26.000342,1671,56.265652,860,39.710421,286,127.205047,...,9.1271,12.3958,57.746243,NaN,38.572100,NaN,25-50%,NaN,25-50%,NaN
1,1,36668,-97.035537,26.071826,1671,50.940346,860,34.179635,286,117.880692,...,1.5259,1.7557,30.571945,NaN,16.120983,NaN,0-25%,NaN,0-25%,NaN
2,2,36669,-96.964053,26.071826,1671,57.783507,860,41.034622,286,124.718795,...,9.1271,12.3958,57.670181,NaN,38.559063,NaN,25-50%,NaN,25-50%,NaN
3,3,36670,-97.035537,26.143310,1671,53.574517,860,37.428918,286,115.740856,...,9.1271,12.3958,32.312887,NaN,14.222778,NaN,0-25%,NaN,0-25%,NaN
4,4,36671,-96.964053,26.143310,1671,60.118581,860,43.774470,286,122.694148,...,9.1271,12.3958,57.596115,NaN,38.506005,NaN,25-50%,NaN,25-50%,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2554,2554,60802,-87.742585,29.717522,1205,90.932440,520,76.494158,1533,86.555143,...,9.9857,11.2027,30.814319,NaN,16.993804,NaN,0-25%,NaN,0-25%,NaN
2555,2555,60808,-87.742585,29.789007,1205,84.443641,520,68.570209,1533,79.735503,...,6.8179,5.9596,30.510465,NaN,15.706714,NaN,0-25%,NaN,0-25%,NaN
2556,2556,60864,-88.314459,30.217912,1497,27.789683,2459,17.932734,576,23.083298,...,3.0173,8.4316,53.280620,NaN,33.239677,NaN,25-50%,NaN,0-25%,NaN
2557,2557,60865,-88.242975,30.217912,1497,34.138032,3674,15.290775,576,20.378109,...,10.4977,13.0178,53.660471,NaN,34.577485,NaN,25-50%,NaN,0-25%,NaN


The data contains 2559 rows describing gridded pixel locaitons across the Gulf of Mexico and 68 features related to each location.

In [3]:
wendt.columns

Index(['Grid FID', 'Grid ID', 'Longitude\n(degrees)', 'Latitude\n(degrees)',
       'CO2 Source ID (tied to NATCARB dataset)*', 'Distance (km)',
       'CO2 Source ID (tied to NATCARB dataset)', 'Distance (km).1',
       'CO2 Source ID (tied to NATCARB dataset).1', 'Distance (km).2',
       'Caisson Platform', 'Fixed Platform', 'Well Protector Platform',
       'Pipeline', 'Fault', 'Salt dome', 'Shipping Route (w/5 MI Buffer)',
       'Shipping Route (w/10 mi Buffer)', 'Fault.1', 'Active Wells',
       'P&A Wells', 'Caisson Platform.1', 'Major Platform',
       'Well Protector Platform.1', 'Pipeline.1', 'Salt dome.1', 'EOR Wells',
       'Shipping Route (w/10 mi Buffer).1', 'Number of plays',
       '(SUM) Injectivity W Depth - (kh)',
       '(SUM) Quality W/O Depth - ([h x E]/[1,000 feet to reservoir bottom])',
       'Capacity per Grid - (million tonnes)', 'Water Depth - (feet)',
       'Average Calculated Water Depth - (feet)',
       '(SUM) Hydrocarbon Potential W/O Depth - ([h x S

# Drop Columns
First, I drop columns in the dataset that contain irrelevant information or logistical information that should not be used when applying machine learning techniques. I drop grid identifier information such as **Grid ID** and **Grid FID**. I also drop **CO2 Source ID (tied to NATCARB dataset)** columns because they contain ID location information and would hinder models in areas with vastly different ID information. I also drop the **Longitude\n(degrees)** and **Latitude\n(degrees)** columns for the same concern of reproducibility. Ideally, machine learning techniques will rely on site features opposed to geospatial locations or codes for modeling and applications. 

In [4]:
wendt = wendt.drop(
    columns = ['Grid ID', 'Grid FID', 'CO2 Source ID (tied to NATCARB dataset)*', 
               'CO2 Source ID (tied to NATCARB dataset)', 'CO2 Source ID (tied to NATCARB dataset).1',
              "Longitude\n(degrees)", "Latitude\n(degrees)"])

# Assess Missing Data
Next, I evaluate potential missing data. 

In [5]:
for column in wendt.columns:
    print(sum(wendt[column].isna()))

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
2233
0
2233
0
2233
0
2233


The only columns with missing data are the site scores and their respective quartiles for Wendt's Scenario 2 and Scenario 4. Since my analysis is centered around Scenarios 1 and 3, there is no relevant missing data. Next, I drop the columns with missing data both because they are missing substantial data and because the scenarios are not relevant.

In [6]:
wendt = wendt.drop(
    columns = ['S2', 'S4', 'S2.1', 'S4.1'])

# Modify Data Type

The last 2 columns contain quartile information for the site score, which is currently stored as a string. I modify the data types to floats (0.25, 0.5, 0.75, 1) to represent the quartile bin.

In [7]:
wendt.iloc[:, -1]

0        25-50%
1         0-25%
2        25-50%
3         0-25%
4        25-50%
         ...   
2554      0-25%
2555      0-25%
2556      0-25%
2557      0-25%
2558    75-100%
Name: S3.1, Length: 2559, dtype: object

To replace the column with float values, I extract the last digit in the bin using regular expressions. I store this as **viability_score1_quartile**. I also merge this back to the original dataframe. I do this for both Scenario 1 and Scenario 3 data.

In [8]:
viability_score1_quartile = wendt['S1.1'].str.extract(f'(\d+)%').astype(int)
viability_score1_quartile.columns = ['Viability_score1_quartile']

viability_score3_quartile = wendt['S3.1'].str.extract(f'(\d+)%').astype(int)
viability_score3_quartile.columns = ['Viability_score3_quartile']

In [9]:
wendt = wendt.merge(viability_score1_quartile, left_index=True, right_index=True)
wendt = wendt.merge(viability_score3_quartile, left_index=True, right_index=True)

Finally, I drop **S1.1** and **S3.1**.

In [10]:
wendt= wendt.drop(columns = ['S1.1', 'S3.1'])

# Save to csv
Now, I save this dataframe as a csv to open in another jupyter notebook.

In [11]:
wendt.to_csv('wendt_df.csv')